# Hurtownia Danych Iowa Liquor Sales: Architektura i Transformacje
Niniejszy notatnik dokumentuje architekturę ETL oraz warstwę analityczną projektu.
Projekt opiera się na architekturze Medallion oraz paradygmacie OLAP (Online Analytical Processing).
Analiza obejmuje 3 warstwy:
1. **Warstwa Staging (Bronze)** - Ekstrakcja danych źródłowych i omówienie problemów z jakością danych.
2. **Warstwa Data Warehouse (Silver)** - Wielowymiarowy model gwiazdy. Obejmuje deduplikację, generowanie kluczy zastępczych (surrogate keys), obsługę braków danych oraz kalkulacje SQL.
3. **Warstwa Semantic (Gold)** - Zbiór 16 zoptymalizowanych widoków SQL pełniących rolę dedykowanych martów danych dla domen: Czas, Geografia, Asortyment, Wskaźniki biznesowe.


In [1]:
import os
import sys
sys.path.append(os.path.abspath('..'))
import pandas as pd
from src.utils.db import sqlserver_connection
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
def query_db(sql_query: str) -> pd.DataFrame:
    with sqlserver_connection() as conn:
        return pd.read_sql(sql_query, conn)


## 1. Warstwa Staging (Bronze) - Przegląd surowych danych
Pierwszym etapem procesu ETL jest ekstrakcja i załadowanie danych do tabeli tymczasowej `stg.iowa_liquor_sales_raw`. 
Poniżej zaprezentowano próbkę wszystkich surowych danych tuż po załadowaniu z API (brak jakichkolwiek modyfikacji):


In [2]:
sql_raw = """
SELECT TOP 3 *
FROM stg.iowa_liquor_sales_raw;
"""
display(query_db(sql_raw))


Powyższe dane posiadają liczne problemy jakościowe. Zanim trafią dalej, poddawane są transformacjom w Pythonie (Pandas):
1. **Parsowanie wartości finansowych**: Usunięcie symboli walut i separatorów tysięcy (np. znak dolara, przecinki) oraz konwersja na typy numeryczne (np. dla `state_bottle_cost` czy `sale_dollars`).
2. **Ekstrakcja współrzędnych**: Wydzielenie długości (`longitude`) i szerokości (`latitude`) geograficznej z formatu tekstowego typu POINT w kolumnie `store_location`.
3. **Deduplikacja (Hashowanie)**: Wyliczenie unikalnego identyfikatora `source_row_hash` za pomocą algorytmu SHA-256 w celu śledzenia zmian w rekordach.
4. **Ładowanie danych**: Zapis wsadowy (batch processing) przy użyciu sterownika ODBC (`cursor.fast_executemany = True`).


## 2. Warstwa Data Warehouse (Silver) - Model Gwiazdy
W tej warstwie dane transformowane są do postaci analitycznej (OLAP). Skrypty SQL odpowiadają za:
### Deduplikacja Wymiarów
Zarówno pliki źródłowe, jak i warstwa staging mogą zawierać zduplikowane wpisy. Aby wygenerować unikalne rekordy w tabelach wymiarów (np. `dim_store`), wykorzystano funkcje okna (Window Functions):
```sql
WITH ranked AS (
    SELECT *, ROW_NUMBER() OVER (
        PARTITION BY COALESCE(NULLIF(store_number, ''), 'UNKNOWN')
        ORDER BY date DESC, staging_key DESC
    ) AS rn
    FROM stg.iowa_liquor_sales_raw
)
INSERT INTO dw.dim_store ... SELECT ... FROM ranked WHERE rn = 1;
```
Klauzula `ROW_NUMBER() ... = 1` gwarantuje zachowanie tylko najnowszego rekordu z systemu źródłowego.
### Obsługa braków danych (Wzorzec 'Unknown Member')
Zgodnie z zasadami modelowania wymiarowego (metodyka Kimballa), tabela faktów nie powinna zawierać wartości NULL w kluczach obcych. Aby zapewnić spójność referencyjną, operacje złączeń wykorzystują funkcję `COALESCE` (np. `COALESCE(NULLIF(raw.category, ''), 'UNKNOWN')`). W przypadku braku dopasowania, fakty przypisywane są do domyślnego rekordu 'UNKNOWN', co pozwala na zachowanie ciągłości kalkulacji finansowych.
Poniżej przedstawiono rezultaty działania tych mechanizmów w warstwie Wymiarów:


In [3]:
print("1. Wymiar Sklepu po ekstrakcji na czyste numeryczne współrzędne Latitude i Longitude:")
display(query_db("SELECT TOP 3 store_key, store_name, latitude, longitude FROM dw.dim_store WHERE latitude IS NOT NULL;"))
print("\n2. Defensywna technika wymiarowa: Rekord UNKNOWN dbający o relacyjną integralność faktów:")
display(query_db("SELECT * FROM dw.dim_category WHERE category_number = 'UNKNOWN';"))


### Tabela Faktów
Tabela `dw.fact_sales` centralizuje dane. Aby odciążyć warstwę aplikacji analitycznych (BI), kluczowe wskaźniki biznesowe (np. marża kwotowa `margin_amount`) są wyliczane i agregowane na poziomie hurtowni danych na etapie tworzenia tabeli faktów.


In [4]:
print("Fakty i pre-kalkulowana stopa zwrotu:")
display(query_db("SELECT TOP 5 invoice_number, store_key, category_key, sale_dollars, state_bottle_cost, margin_amount FROM dw.fact_sales;"))


## 3. Warstwa Semantic (Gold) - Datamarts
Warstwa semantyczna stanowi interfejs dostępowy dla narzędzi analitycznych i raportowych. Zastosowanie dedykowanych widoków realizuje następujące założenia:
1. **Single Source of Truth**: Kluczowe wskaźniki efektywności (KPI) są zdefiniowane jednoznacznie na poziomie bazy danych, co eliminuje niespójności w raportowaniu między różnymi narzędziami BI.
2. **Abstrakcja logiki złączeń**: Narzędzia raportowe odpytują pojedyncze, płaskie widoki (`vw_*`), bez konieczności odtwarzania złożonych relacji (JOIN) między wymiarami a faktami.
3. **Segmentacja logiczna**: Utworzono 16 tematycznych widoków danych (data marts) pogrupowanych w domeny analityczne.
Poniżej zaprezentowano przykłady widoków.


### Grupa 1: Ogólny Status i Wskaźniki Biznesowe (High-Level/ETL)
Zbiory stworzone pod użytek Zarządu (CEO, CFO) oraz zespołu technicznego Data Engineering (do audytu bazy).
#### 1. `sem.vw_kpi_summary`
Silnie zagregowany widok biznesowy obliczający uniwersalne wskaźniki (KPI - Key Performance Indicators) całego podmiotu. Liczy on na poziomie silnika bazodanowego średnie przychody, procentowe marże operacyjne oraz wolumen litrażu.


In [5]:
### Grupa 1: Ogólny Status i Wskaźniki Biznesowe
Widoki agregujące dane na najwyższym poziomie ogólności, przeznaczone dla celów zarządczych oraz monitorowania procesu ETL.
#### 1. `sem.vw_kpi_summary`
Widok kalkulujący podstawowe wskaźniki (KPI) dla całego zbioru danych. Obejmuje m.in. całkowite przychody, uśrednione marże oraz wolumen sprzedaży.


/tmp/ipykernel_1294/1681752413.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql_query, conn)


,total_sales,total_margin,sales_line_count,total_bottles_sold,total_volume_liters,invoice_count,store_count,product_count,category_count,vendor_count,avg_invoice_value,avg_bottles_per_invoice,avg_margin_percent,sales_per_store,sales_per_liter
0,4.466417e+08,1.492504e+08,2635879,31302201.0,23756277.41,2635879,2110,5231,48,250,169.446953,11.875431,33.42,211678.514578,18.800995


#### 2. `sem.vw_etl_status`
Diagnostyka potoku ETL pokazująca aktualny stan zapełnienia (liczba rekordów) tabeli faktów i wszystkich wymiarów. Konieczna przy porannych przeglądach technicznych by natychmiast wychwycić błędy w zasileniach DWH.


In [6]:
#### 2. `sem.vw_etl_status`
Widok diagnostyczny przedstawiający liczbę rekordów w poszczególnych tabelach (faktach i wymiarach) oraz znaczniki czasowe ostatnich ładowań danych. Używany do monitorowania poprawności działania potoku ETL.


/tmp/ipykernel_1294/1681752413.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql_query, conn)


,status_generated_at,staging_row_count,fact_row_count,dim_date_count,dim_store_count,dim_product_count,dim_category_count,dim_vendor_count,dim_packaging_count,min_date,max_date,last_staging_load_timestamp,last_fact_load_timestamp
0,2026-07-04 21:38:25.553333,2639557,2635879,290,2110,5231,48,250,69,2023-01-02,2023-12-30,2026-07-04 19:53:26.060703,2026-07-04 19:53:43.247454


### Grupa 2: Domenowy Wymiar Czasu (Time-Series Analysis)
Widoki grupujące fakty ściśle po osiach temporalnych. Wymiar daty `dim_date` to silne narzędzie Kimballa pozwalające uniknąć stosowania dziesiątek funkcji operujących na dacie w bazowym SQL-u.
#### 3. `sem.vw_sales_overview`
Gigantyczny płaski widok ogólny. Wycina i zastępuje w pełni techniczne klucze sztuczne (Surrogate Keys, te kończące się na `_key`) prawdziwymi wartościami zrozumiałymi dla człowieka, dając możliwość analizowania wszystkich trendów czasowych na poziomie wiersza, z uwzględnieniem geolokalizacji.


In [7]:
### Grupa 2: Analiza w czasie (Wymiar Daty)
Grupowanie zdarzeń sprzedażowych w ujęciu temporalnym. Wykorzystanie ujednoliconego wymiaru daty (`dim_date`) upraszcza zapytania analityczne.
#### 3. `sem.vw_sales_overview`
Szczegółowy widok denormalizujący klucze zastępcze (surrogate keys) na rzecz wartości opisowych. Pozwala na analizę transakcji na poziomie pojedynczego wiersza z uwzględnieniem atrybutów czasu i lokalizacji.


/tmp/ipykernel_1294/1681752413.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql_query, conn)


,date,invoice_number,store_name,category_name,sale_dollars
0,2023-01-02,INV-54554000001,CENTRAL CITY 2,100% AGAVE TEQUILA,261.00
1,2023-01-02,INV-54554000002,CENTRAL CITY 2,AMERICAN VODKAS,418.80
2,2023-01-02,INV-54554000003,CENTRAL CITY 2,IMPORTED FLAVORED VODKA,358.56
3,2023-01-02,INV-54554000004,CENTRAL CITY 2,CREAM LIQUEURS,306.00
4,2023-01-02,INV-54554000005,CENTRAL CITY 2,SPICED RUM,1124.40


#### 4. `sem.vw_sales_by_day_type`
Wybitne zastosowanie inteligentnych struktur w Data Warehouse. Narzędzie grupuje zysk w oparciu o przypisaną do wymiaru czasu flagę `is_weekend`, dając błyskawiczną odpowiedź (zsumowane wartości finansowe) dla analizy różnicy zapotrzebowania w wolne dni rynkowe.


In [8]:
display(query_db("SELECT * FROM sem.vw_sales_by_day_type;"))


/tmp/ipykernel_1294/1681752413.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql_query, conn)


,year,quarter,month,year_month,is_weekend,day_type,total_sales,total_bottles_sold,total_volume_liters,total_margin,sales_line_count,invoice_count
0,2023,3,7,2023-07,True,Weekend,1186208.53,87163.0,61733.02,395977.14,8854,8854
1,2023,1,1,2023-01,True,Weekend,838436.05,66807.0,45927.71,279843.16,7061,7061
2,2023,2,6,2023-06,True,Weekend,1424375.15,105970.0,76931.37,474511.15,9701,9701
3,2023,3,9,2023-09,True,Weekend,1728305.20,128850.0,95695.29,577184.04,12263,12263
4,2023,3,8,2023-08,True,Weekend,298.45,12.0,15.00,102.50,7,7
5,2023,4,11,2023-11,True,Weekend,3412682.64,271801.0,178442.29,1138208.97,20659,20659
6,2023,1,3,2023-03,False,Weekday,36433363.96,2632359.0,2016057.37,12183998.11,221299,221299
7,2023,2,5,2023-05,False,Weekday,39704733.36,2784835.0,2186163.82,13392595.99,232960,232960
8,2023,4,11,2023-11,False,Weekday,35929764.34,2395754.0,1884156.66,11984428.10,202653,202653
9,2023,4,12,2023-12,True,Weekend,3407799.86,262935.0,170366.75,1136876.98,24738,24738


#### 5. `sem.vw_sales_by_month`
Najpopularniejsza, uniwersalna struktura w analityce OLAP: Agregacja w hierarchii Rok -> Miesiąc. Widok gotowy, by rzucić go na wykres liniowy w BI (np. w używanym przez nas frameworku Streamlit).


In [9]:
display(query_db("SELECT TOP 5 * FROM sem.vw_sales_by_month ORDER BY year, month;"))


/tmp/ipykernel_1294/1681752413.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql_query, conn)


,year,quarter,month,year_month,total_sales,total_bottles_sold,total_volume_liters,total_margin,sales_line_count,invoice_count,store_count
0,2023,1,1,2023-01,32582340.63,2358449.0,1747102.70,10888433.83,212850,212850,1857
1,2023,1,2,2023-02,32134462.65,2289374.0,1771119.28,10750429.35,189294,189294,1824
2,2023,1,3,2023-03,36436060.72,2632521.0,2016226.87,12184903.75,221313,221313,1859
3,2023,2,4,2023-04,32915910.22,2393665.0,1793682.48,10986201.79,198446,198446,1832
4,2023,2,5,2023-05,39721449.29,2786013.0,2187271.06,13398186.64,233060,233060,1879


#### 6. `sem.vw_avg_sales_per_store_by_month_region`
Krzyżuje strukturę czasu ze strukturą przestrzenną. Odpowiada na rozbudowane, trudne zadania zarządcze: "Ile średnio (Average) na punkt przynosi miesiąc x w Hrabstwie y?".


In [10]:
display(query_db("SELECT TOP 5 * FROM sem.vw_avg_sales_per_store_by_month_region;"))


/tmp/ipykernel_1294/1681752413.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql_query, conn)


,year,quarter,month,year_month,state_name,county,city,store_count,avg_sales_per_store,avg_bottles_per_store,avg_volume_liters_per_store,avg_margin_per_store
0,2023,4,11,2023-11,Iowa,Polk,Des Moines,84,54498.716071,3962.452380,2617.673452,18168.369880
1,2023,1,3,2023-03,Iowa,Monona,Onawa,4,10462.017500,903.250000,547.420000,3500.330000
2,2023,2,4,2023-04,Iowa,Mills,Pacific Junction,2,4038.905000,539.000000,203.310000,1346.735000
3,2023,1,3,2023-03,Iowa,Cass,Atlantic,7,14203.137142,955.714285,797.464285,4753.011428
4,2023,3,9,2023-09,Iowa,Cerro Gordo,Mason City,17,26915.348235,2127.411764,1585.295294,8976.100588


### Grupa 3: Domenowy Wymiar Asortymentu i Opłacalności (Products & Margins)
Wyliczenia zysku (rentowności) w rozbiciu na dostawców, kategorie produktów i rozmiary dystrybuowanych opakowań.
#### 7. `sem.vw_sales_by_category`
Złoty klasyk agregacji. Warto tu zaznaczyć inżynieryjne zastosowanie `CROSS JOIN` ze zliczoną wcześniej całkowitą wartością przychodów (totals), w celu pre-kalkulacji parametru `sales_share_percent`! System analityczny nie musi dzielić na "swoim domowym kalkulatorze" części przez sumę, dostaje obrobiony wynik bezpośrednio z warstwy semantycznej.


In [11]:
display(query_db("SELECT TOP 5 category_name, total_sales, sales_share_percent, avg_margin_per_bottle FROM sem.vw_sales_by_category ORDER BY total_sales DESC;"))


/tmp/ipykernel_1294/1681752413.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql_query, conn)


,category_name,total_sales,sales_share_percent,avg_margin_per_bottle
0,AMERICAN VODKAS,67412480.47,15.09,3.453703
1,CANADIAN WHISKIES,50411669.32,11.29,5.498948
2,STRAIGHT BOURBON WHISKIES,38747595.32,8.68,7.536290
3,100% AGAVE TEQUILA,32384034.97,7.25,9.742586
4,WHISKEY LIQUEUR,26564898.28,5.95,2.004608


#### 8. `sem.vw_category_sales_over_time`
Widok wykorzystywany głównie w uczeniu maszynowym do wykrywania trendów rynkowych, spadków i analizy predykcyjnej, wiążący wskaźnik obrotu na produkcie z siatką czasową rynkową.


In [12]:
display(query_db("SELECT TOP 5 * FROM sem.vw_category_sales_over_time;"))


/tmp/ipykernel_1294/1681752413.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql_query, conn)


,year,quarter,month,year_month,category_name,total_sales,total_bottles_sold,total_volume_liters,total_margin,sales_line_count,invoice_count,store_count
0,2023,2,5,2023-05,COCKTAILS/RTD,828160.91,70122.0,99899.43,276095.97,8633,8633,993
1,2023,1,2,2023-02,IMPORTED BRANDIES,940218.52,40581.0,18497.41,313499.29,3378,3378,895
2,2023,2,4,2023-04,CANADIAN WHISKIES,3393790.41,210239.0,190317.16,1131944.35,18939,18939,1752
3,2023,3,8,2023-08,CANADIAN WHISKIES,3597054.97,231873.0,207808.41,1199436.73,21331,21331,1812
4,2023,4,12,2023-12,SCOTCH WHISKIES,703099.24,21524.0,21266.74,234478.07,3199,3199,725


#### 9. `sem.vw_top_products`
Tabela dedykowana odpytywaniu na bardzo głębokim ziarnie produktowym. Klasyczny ranking "Najlepiej sprzedający się konkretny produkt/kod" wsparty wolumenem litrażu (szczególnie krytycznym z perspektywy optymalizacji logistyki wysyłek).


In [13]:
display(query_db("SELECT TOP 5 item_description, total_bottles_sold, total_sales FROM sem.vw_top_products ORDER BY total_sales DESC;"))


/tmp/ipykernel_1294/1681752413.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql_query, conn)


,item_description,total_bottles_sold,total_sales
0,TITOS HANDMADE VODKA,400398.0,11411343.00
1,TITOS HANDMADE VODKA,506485.0,10008143.60
2,BLACK VELVET,524325.0,8567077.98
3,CAPTAIN MORGAN ORIGINAL SPICED BARREL,256199.0,7063866.82
4,TITOS HANDMADE VODKA,438152.0,6572280.00


#### 10. `sem.vw_margin_analysis`
Filar opłacalności naszego biznesu. Uśredniony koszt zakupu hurtowego zderzony ze stawką dystrybucyjną rynkową by ujawnić marżę na pojedynczą jednostkę towarową (butelkę). Skalowane do totalnych sum dla analiz rentowności dostaw.


In [14]:
display(query_db("SELECT TOP 5 category_name, vendor_name, avg_unit_margin, total_margin FROM sem.vw_margin_analysis ORDER BY total_margin DESC;"))


/tmp/ipykernel_1294/1681752413.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql_query, conn)


,category_name,vendor_name,avg_unit_margin,total_margin
0,AMERICAN VODKAS,FIFTH GENERATION INC,5.734500,10027926.96
1,CANADIAN WHISKIES,HEAVEN HILL BRANDS,3.629779,4119795.00
2,WHISKEY LIQUEUR,SAZERAC COMPANY INC,2.815516,3879317.94
3,CANADIAN WHISKIES,DIAGEO AMERICAS,8.952332,3410927.00
4,TENNESSEE WHISKIES,BROWN FORMAN CORP.,9.499180,3211062.02


#### 11. `sem.vw_sales_by_packaging`
Agreguje rynkowe przychody do grup wolumenowych pojemników, by wyliczyć optymalne wielkości zamawianych zapasów od dostawców (koreluje z naszymi wewnętrznymi klasyfikatorami małpka/standardówka, zaszytymi w wymiarze `dim_packaging`).


In [15]:
display(query_db("SELECT TOP 5 * FROM sem.vw_sales_by_packaging ORDER BY total_sales DESC;"))


/tmp/ipykernel_1294/1681752413.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql_query, conn)


,pack,bottle_volume_ml,volume_group,total_sales,total_bottles_sold,total_volume_liters,total_margin,sales_line_count,invoice_count,sales_share_percent
0,12,750,standard,1.316039e+08,8054198.0,6040648.50,43918698.82,828691,828691,29.47
1,6,1750,extra_large,1.020161e+08,5198080.0,9096640.00,34084839.23,457790,457790,22.84
2,12,1000,large,7.177420e+07,4537105.0,4537105.00,23966457.47,226615,226615,16.07
3,6,750,standard,6.171046e+07,1968954.0,1476715.50,20602243.01,332328,332328,13.82
4,24,375,small,2.056612e+07,3411086.0,1279015.35,6867397.04,201109,201109,4.60


#### 12. `sem.vw_sales_by_vendor`
Rozliczenie kwot, prowizji i wykaz obrotów handlowych spójnych z gigantycznymi dostawcami alkoholu dostarczającymi zamówienia do Iowa (np. Diageo, Sazerac).


In [16]:
display(query_db("SELECT TOP 5 * FROM sem.vw_sales_by_vendor ORDER BY total_sales DESC;"))


/tmp/ipykernel_1294/1681752413.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql_query, conn)


,vendor_name,total_sales,total_bottles_sold,total_margin,sales_line_count,sales_share_percent
0,DIAGEO AMERICAS,89105882.60,4237650.0,29748794.14,393222,19.95
1,SAZERAC COMPANY INC,66845842.81,8381527.0,22446497.38,454451,14.97
2,FIFTH GENERATION INC,30880585.64,1672757.0,10295491.76,88069,6.91
3,JIM BEAM BRANDS,30077014.54,1879448.0,10034039.49,205949,6.73
4,PERNOD RICARD USA,27732294.21,1369487.0,9252738.93,133186,6.21


### Grupa 4: Domenowy Wymiar Przestrzenny (Geospatial & Stores)
Mapowanie danych sprzedażowych na konkretne koordynaty, sklepy i hrabstwa pozwalające na tworzenie tzw. heat-map.
#### 13. `sem.vw_sales_by_geography`
Górny Roll-up terytorialny. Pozwala na budowanie kartogramów bazujących na miastach oraz przynależnych do nich Hrabstwach, umożliwiając wyodrębnienie obszarów o silnej opłacalności wewnątrzkrajowej.


In [17]:
display(query_db("SELECT TOP 5 county, city, total_sales, total_bottles_sold FROM sem.vw_sales_by_geography ORDER BY total_sales DESC;"))


/tmp/ipykernel_1294/1681752413.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql_query, conn)


,county,city,total_sales,total_bottles_sold
0,Polk,Des Moines,54716593.67,3964525.0
1,Linn,Cedar Rapids,28165657.82,2047863.0
2,Scott,Davenport,20903373.82,1700718.0
3,Pottawattamie,Council Bluffs,15282146.05,1140320.0
4,Woodbury,Sioux City,14566939.00,1065968.0


#### 14. `sem.vw_sales_map_points`
Prawdziwe dane wejściowe dla bibliotek GIS-owych. Widok dba o to by nie doszło do rzucenia wyjątkiem na warstwie wizualizacyjnej ze względu na złe dane - wymusza silnym filtrem z SQL `WHERE latitude IS NOT NULL AND longitude IS NOT NULL` tylko bezbłędne ukształtowanie rekordów.


In [18]:
display(query_db("SELECT TOP 5 store_name, city, latitude, longitude, total_sales FROM sem.vw_sales_map_points ORDER BY total_sales DESC;"))


/tmp/ipykernel_1294/1681752413.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql_query, conn)


,store_name,city,latitude,longitude,total_sales
0,HY-VEE #3 / BDI / DES MOINES,Des Moines,41.554269,-93.594781,15275187.46
1,CENTRAL CITY 2,Des Moines,41.605835,-93.613286,13740361.39
2,ANOTHER ROUND / DEWITT,Dewitt,41.809631,-90.538996,6894898.17
3,HY-VEE WINE AND SPIRITS #1 (1281) / IOWA CITY,Iowa City,41.642516,-91.529426,6112597.82
4,BENZ DISTRIBUTING,Cedar Rapids,41.975513,-91.659640,5235975.06


#### 15. `sem.vw_sales_by_store`
Zestawienie indywidualnej efektywności finansowej konkretnych punktów handlu. Bazy informacyjne pod wypłaty i premie dla operatorów sklepów.


In [19]:
display(query_db("SELECT TOP 5 store_name, total_sales, total_margin, invoice_count FROM sem.vw_sales_by_store ORDER BY total_sales DESC;"))


/tmp/ipykernel_1294/1681752413.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql_query, conn)


,store_name,total_sales,total_margin,invoice_count
0,HY-VEE #3 / BDI / DES MOINES,15275187.46,5098821.19,20611
1,CENTRAL CITY 2,13740361.39,4586639.30,20506
2,ANOTHER ROUND / DEWITT,6894898.17,2301835.13,11761
3,HY-VEE WINE AND SPIRITS #1 (1281) / IOWA CITY,6112597.82,2040235.09,10041
4,BENZ DISTRIBUTING,5235975.06,1748162.77,14760


#### 16. `sem.vw_volume_vs_revenue`
Zaawansowana analityka wielowymiarowa. Poszukuje korelacji gęstości dostaw transportu mierzonych w ciężarach / uciągach (Gallons, Liters) w konfrontacji do zwracanych marż finansowych, aby wesprzeć algorytmy decyzyjne zespołu Supply Chain / łańcucha dostaw.


In [20]:
display(query_db("SELECT TOP 5 city, total_volume_liters, total_sales, sales_per_liter FROM sem.vw_volume_vs_revenue ORDER BY total_sales DESC;"))


/tmp/ipykernel_1294/1681752413.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql_query, conn)


,city,total_volume_liters,total_sales,sales_per_liter
0,Des Moines,2628116.69,54716593.67,20.819697
1,Cedar Rapids,1479865.42,28165657.82,19.032580
2,Davenport,1119286.67,20903373.82,18.675621
3,Council Bluffs,772569.21,15282146.05,19.780941
4,Sioux City,760230.63,14566939.00,19.161210
